# tanish-ml-model-demo



## Install dependencies

In [0]:
%pip install -r requirements.txt

## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '487rklshfls39d'
os.environ['DataZoneDomainId'] = 'dzd-ctwqq3wi20ovk1'
os.environ['DataZoneEnvironmentId'] = '5i7v2zg7ak7xq9'
os.environ['DataZoneDomainRegion'] = 'ap-south-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "487rklshfls39d",
                "DataZoneDomainId": "dzd-ctwqq3wi20ovk1",
                "DataZoneEnvironmentId": "5i7v2zg7ak7xq9",
                "DataZoneDomainRegion": "ap-south-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import joblib

# Load data
X, y = load_iris(return_X_y=True)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X, y)

# Save model
joblib.dump(model, "model.joblib")

print("Model trained!")

Model trained!


In [0]:
import boto3

s3 = boto3.client('s3')

bucket = "demo-ml-tanish"   # create bucket in S3 console
s3.upload_file("model.joblib", bucket, "model/model.joblib")

print("Uploaded to S3")

Uploaded to S3


In [0]:
with open("inference.py", "w") as f:
    f.write("""
import joblib
import json

def model_fn(model_dir):
    return joblib.load(model_dir + "/model.joblib")

def predict_fn(input_data, model):
    return model.predict(input_data)

def input_fn(request_body, request_content_type):
    return json.loads(request_body)

def output_fn(prediction, content_type):
    return json.dumps(prediction.tolist())
""")

In [0]:
import os
print(os.listdir())

['.bashrc', '.ipython', '.user_packages', '.cache', 'model.joblib', 'inference.py']


In [0]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib")

print("Tar file created!")

Tar file created!


In [0]:
s3.upload_file("model.tar.gz", bucket, "model/model.tar.gz")
print("Uploaded tar.gz to S3")

Uploaded tar.gz to S3


In [0]:
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

role = sagemaker.get_execution_role()
session = sagemaker.Session(default_bucket="demo-ml-tanish")

model = SKLearnModel(
    model_data=f"s3://demo-ml-tanish/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    source_dir=".",
    framework_version="1.0-1",
    sagemaker_session=session
)

predictor = model.deploy(
    instance_type="ml.t2.medium",
    initial_instance_count=1
)

print("Model deployed successfully!")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


-

-

...

-

-

-

!

Model deployed successfully!


In [0]:
import json
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

result = predictor.predict([[5.1, 3.5, 1.4, 0.2]])
print(f"Prediction: {result}")

Prediction: [0]


In [0]:
## this code for the lambda function 
""" import json
import boto3

runtime = boto3.client("sagemaker-runtime")

ENDPOINT_NAME = "your-endpoint-name"  <----------change  here 

def lambda_handler(event, context):
    if "body" in event:
        body = json.loads(event["body"])
    else:
        body = event

    data = body.get("input")

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(data)
    )

    result = response["Body"].read().decode()

    return {
        "statusCode": 200,
        "body": result
    }"""

In [0]:
## ----> for api test --->curl -X POST "https://your-api-id.execute-api.ap-south-1.amazonaws.com/predict" -H "Content-Type: application/json" -d "{\"input\":[[1200,2]]}"

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()